In [ ]:
from project.el5600 import project
ic = project(device="el5600", revision="a1", emulator="cp2112", logging=False, is_gui=False)

from phy.multimeter import siglent_sdm3055_auto
from phy.power_supply import keithley_2460, rigol_dp821a, keysight_e36232a
from phy.scope import tektronix_dpo4104b_usb
from phy.eloader import sdl1020x
from phy.relay_16ch import relay_box
# from phy.chamber_th3 import chamber

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

from time import sleep as delay
import numpy as np

chart = plot()

dm = siglent_sdm3055_auto()
ps = rigol_dp821a()
ks = keysight_e36232a()
kt = keithley_2460()
ld = sdl1020x()
sc = tektronix_dpo4104b_usb()

# dma = agilent_34401a("COM5")
relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)


# ==================================
def disable_all_ps(kt=kt, ps=ps):
    kt.disable
    ld.disable
    ks.disable
    ps.ch1.disable
    ps.ch2.disable
# ==================================

# Test A13
- e36232 : direct to VIN
- kt : VDD3V3
- ps.ch1 : relay
- ps.ch2 : to relay.ch3 for EN
- relay.ch2 : VOUT
- relay.ch3 : EN
- loader : to relay.ch2

[SS_TIM]
0b : 1V/ms
1b : 2.5V/ms

In [ ]:
disable_all_ps()
relay.init_channels

v_vdd = 3.3
v_en = 3.3

# --------------------------------------------
ps.ch1.cfg_all = 5, 1 # relay
ps.ch1.enable

# --------------------------------------------
kt.cfg_all = v_vdd, 0.01 # vdd
kt.enable
delay(0.5)

# --------------------------------------------
ps.ch2.cfg_all = v_en, 0.1
ps.ch2.enable
delay(0.5)
relay.ch3.enable # en
delay(0.5)

# --------------------------------------------
ks.cfg_all = 5, 1
ks.power_recycle

# --------------------------------------------
relay.enable = 1, 2



In [ ]:
list_vin = [5, 20, 28]

for n in range(2):
    
    ic.SS_TIME = n
    
    for voltage in list_vin:
        
        sc.ch3.trigger_rise = 1.5
        
        if voltage == 5:
            sc.ch2.vertical_scale = 1
            sc.horizontal_scale = 1e-3
        elif voltage == 20:
            sc.ch2.vertical_scale = 5
            sc.horizontal_scale = 4e-3
        elif voltage == 28:
            sc.ch2.vertical_scale = 5
            sc.horizontal_scale = 4e-3
            
        sc.horizontal_grid = -3
        
        ks.cfg_all = voltage, 0.5
        
        sc.single
        relay.ch3.disable
        ld.enable
        delay(0.5)
        ld.disable
        relay.ch3.enable
        
        sc.save_waveform = f"[25C] t_ON_SS, VIN={voltage:.01f}, SS_TIME={n}"
        
disable_all_ps()